In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
import time
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

df = pd.read_csv('df_final.csv')

dist_cols = [
    'dist_to_city_center', 'dist_to_school', 'dist_to_kindergarten',
    'dist_to_park', 'dist_to_bus_stop', 'dist_to_supermarket'
]
for col in dist_cols:
    df[f'{col}_log'] = np.log1p(df[col])

cols_to_drop = dist_cols + ['Площадь', 'Дата публикации',
                             'Цена', 'Цена_log',
                             'Цена_за_квадратный_метр', 'Цена_за_квадратный_метр_log']
X = df.drop(columns=cols_to_drop)
y = df['Цена_за_квадратный_метр_log'].values.astype(np.float32)

X = pd.get_dummies(X, columns=['Город', 'Субъект РФ'], drop_first=True)
X = X.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc  = scaler.transform(X_test).astype(np.float32)

INPUT_DIM = X_train_sc.shape[1]
print(f"Input dim: {INPUT_DIM}, train: {len(X_train)}, test: {len(X_test)}")

Input dim: 214, train: 14007, test: 3502


In [2]:
def make_loaders(X_tr, y_tr, X_te, y_te, batch=256):
    tr_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr).unsqueeze(1))
    te_ds = TensorDataset(torch.tensor(X_te), torch.tensor(y_te).unsqueeze(1))
    return (DataLoader(tr_ds, batch_size=batch, shuffle=True),
            DataLoader(te_ds, batch_size=batch*4))

def eval_metrics(model, loader, device='cpu'):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(device)).cpu().numpy())
            trues.append(yb.numpy())
    p = np.concatenate(preds).ravel()
    t = np.concatenate(trues).ravel()
    p_real = np.expm1(p)
    t_real = np.expm1(t)
    return {
        'MAE':  mean_absolute_error(t_real, p_real),
        'MAPE': mean_absolute_percentage_error(t_real, p_real) * 100,
        'R2':   r2_score(t_real, p_real)
    }

def train_model(model, tr_loader, te_loader, epochs=100, lr=1e-3, device='cpu',
                patience=15, scheduler_step=30):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=scheduler_step, gamma=0.5)
    criterion = nn.HuberLoss(delta=0.5)

    best_val, best_state, no_imp = np.inf, None, 0
    t0 = time.time()

    for ep in range(1, epochs+1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in te_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(te_loader.dataset)

        if val_loss < best_val - 1e-5:
            best_val, no_imp = val_loss, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"  Early stop at epoch {ep}")
                break

    model.load_state_dict(best_state)
    elapsed = time.time() - t0
    return elapsed

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ЭКСПЕРИМЕНТ 1: Простая MLP (shallow)
# ═══════════════════════════════════════════════════════════════════════════════
class SimpleMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, 32),     nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.net(x)

print("\n=== Эксперимент 1: SimpleMLP (64-32) ===")
tr_l, te_l = make_loaders(X_train_sc, y_train, X_test_sc, y_test, batch=256)
m1 = SimpleMLP(INPUT_DIM)
t1 = train_model(m1, tr_l, te_l, epochs=200, lr=1e-3, patience=20)
m1_metrics = eval_metrics(m1, te_l)
print(f"MAE: {m1_metrics['MAE']:.0f}  MAPE: {m1_metrics['MAPE']:.2f}%  R2: {m1_metrics['R2']:.4f}  Time: {t1:.1f}s")


=== Эксперимент 1: SimpleMLP (64-32) ===
  Early stop at epoch 142
MAE: 7585  MAPE: 5.65%  R2: 0.8863  Time: 17.7s


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ЭКСПЕРИМЕНТ 2: Deep MLP с BatchNorm + Dropout
# ═══════════════════════════════════════════════════════════════════════════════
class DeepMLP(nn.Module):
    def __init__(self, in_dim, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

print("\n=== Эксперимент 2: DeepMLP (256-128-64-32) + BN + Dropout(0.3) ===")
tr_l, te_l = make_loaders(X_train_sc, y_train, X_test_sc, y_test, batch=256)
m2 = DeepMLP(INPUT_DIM)
t2 = train_model(m2, tr_l, te_l, epochs=300, lr=1e-3, patience=25)
m2_metrics = eval_metrics(m2, te_l)
print(f"MAE: {m2_metrics['MAE']:.0f}  MAPE: {m2_metrics['MAPE']:.2f}%  R2: {m2_metrics['R2']:.4f}  Time: {t2:.1f}s")


=== Эксперимент 2: DeepMLP (256-128-64-32) + BN + Dropout(0.3) ===
  Early stop at epoch 108
MAE: 25649  MAPE: 17.48%  R2: 0.4041  Time: 28.8s


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ЭКСПЕРИМЕНТ 3: Deep MLP + Residual connections
# ═══════════════════════════════════════════════════════════════════════════════
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
        self.act = nn.ReLU()
    def forward(self, x): return self.act(x + self.block(x))

class ResidualMLP(nn.Module):
    def __init__(self, in_dim, dim=256, n_blocks=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(in_dim, dim), nn.ReLU())
        self.blocks = nn.Sequential(*[ResBlock(dim, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):
        x = self.input_proj(x)
        x = self.blocks(x)
        return self.head(x)

print("\n=== Эксперимент 3: ResidualMLP (dim=256, 3 блока) ===")
tr_l, te_l = make_loaders(X_train_sc, y_train, X_test_sc, y_test, batch=256)
m3 = ResidualMLP(INPUT_DIM)
t3 = train_model(m3, tr_l, te_l, epochs=300, lr=1e-3, patience=25)
m3_metrics = eval_metrics(m3, te_l)
print(f"MAE: {m3_metrics['MAE']:.0f}  MAPE: {m3_metrics['MAPE']:.2f}%  R2: {m3_metrics['R2']:.4f}  Time: {t3:.1f}s")


=== Эксперимент 3: ResidualMLP (dim=256, 3 блока) ===
  Early stop at epoch 205
MAE: 5673  MAPE: 4.16%  R2: 0.9430  Time: 90.0s


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ЭКСПЕРИМЕНТ 4: ResidualMLP larger
# ═══════════════════════════════════════════════════════════════════════════════
print("\n=== Эксперимент 4: ResidualMLP larger (dim=512, 4 блока, dropout=0.15) ===")
tr_l, te_l = make_loaders(X_train_sc, y_train, X_test_sc, y_test, batch=512)
m4 = ResidualMLP(INPUT_DIM, dim=512, n_blocks=4, dropout=0.15)
t4 = train_model(m4, tr_l, te_l, epochs=300, lr=5e-4, patience=30, scheduler_step=40)
m4_metrics = eval_metrics(m4, te_l)
print(f"MAE: {m4_metrics['MAE']:.0f}  MAPE: {m4_metrics['MAPE']:.2f}%  R2: {m4_metrics['R2']:.4f}  Time: {t4:.1f}s")


=== Эксперимент 4: ResidualMLP larger (dim=512, 4 блока, dropout=0.15) ===
  Early stop at epoch 260
MAE: 5459  MAPE: 3.99%  R2: 0.9433  Time: 233.4s


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
#  ЭКСПЕРИМЕНТ 5: TabNet
# ═══════════════════════════════════════════════════════════════════════════════
from pytorch_tabnet.tab_model import TabNetRegressor

print("\n=== Эксперимент 5: TabNet ===")
t0 = time.time()
tabnet = TabNetRegressor(
    n_d=32, n_a=32,
    n_steps=5,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    seed=SEED,
    verbose=0
)
tabnet.fit(
    X_train_sc, y_train.reshape(-1, 1),
    eval_set=[(X_test_sc, y_test.reshape(-1, 1))],
    eval_metric=['rmse'],
    max_epochs=200,
    patience=20,
    batch_size=256,
    virtual_batch_size=128,
)
t5 = time.time() - t0

y_pred_log = tabnet.predict(X_test_sc).ravel()
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)
m5_metrics = {
    'MAE':  mean_absolute_error(y_test_real, y_pred_real),
    'MAPE': mean_absolute_percentage_error(y_test_real, y_pred_real) * 100,
    'R2':   r2_score(y_test_real, y_pred_real)
}
print(f"MAE: {m5_metrics['MAE']:.0f}  MAPE: {m5_metrics['MAPE']:.2f}%  R2: {m5_metrics['R2']:.4f}  Time: {t5:.1f}s")


=== Эксперимент 5: TabNet ===

Early stopping occurred at epoch 54 with best_epoch = 34 and best_val_0_rmse = 0.14973999559879303
MAE: 15257  MAPE: 11.48%  R2: 0.7375  Time: 95.7s


In [10]:
# ─── Сводная таблица ─────────────────────────────────────────────────────────
print("\n\n" + "="*70)
print("СВОДНАЯ ТАБЛИЦА DL-ЭКСПЕРИМЕНТОВ")
print("="*70)
rows = [
    ("SimpleMLP", "Linear(128→64→32→1)", "lr=1e-3, bs=256, epochs≤200", t1, m1_metrics),
    ("DeepMLP+BN+Dropout", "Linear(256→128→64→32→1)+BN+Drop(0.3)", "lr=1e-3, bs=256, epochs≤300", t2, m2_metrics),
    ("ResidualMLP-S", "proj→3×ResBlock(dim=256)+head", "lr=1e-3, bs=256, epochs≤300", t3, m3_metrics),
    ("ResidualMLP-L", "proj→4×ResBlock(dim=512)+head", "lr=5e-4, bs=512, epochs≤300", t4, m4_metrics),
    ("TabNet", "n_d=32,n_a=32,steps=5,gamma=1.5", "epochs≤200,bs=256,pat=20", t5, m5_metrics),
]
for name, arch, hp, t, m in rows:
    print(f"{name:<22} | MAE={m['MAE']:>7.0f} | MAPE={m['MAPE']:>5.2f}% | R2={m['R2']:.4f} | time={t:>6.1f}s")



СВОДНАЯ ТАБЛИЦА DL-ЭКСПЕРИМЕНТОВ
SimpleMLP              | MAE=   7585 | MAPE= 5.65% | R2=0.8863 | time=  17.7s
DeepMLP+BN+Dropout     | MAE=  25649 | MAPE=17.48% | R2=0.4041 | time=  28.8s
ResidualMLP-S          | MAE=   5673 | MAPE= 4.16% | R2=0.9430 | time=  90.0s
ResidualMLP-L          | MAE=   5459 | MAPE= 3.99% | R2=0.9433 | time= 233.4s
TabNet                 | MAE=  15257 | MAPE=11.48% | R2=0.7375 | time=  95.7s


# Лучшее DL-решение: ResidualMLP-L

Лучшей архитектурой стала ResidualMLP-L — глубокая полносвязная сеть с остаточными связями. Архитектурно она устроена так: входной проекционный слой (214 → 512) + 4 резидуальных блока (каждый: Linear → GELU → Dropout → Linear → сумма со входом) + выходная голова (512 → 64 → 1).

Итоговые результаты: MAE = 5 459 руб/м², MAPE = 3.99%, R² = 0.9433.

# Почему одна архитектура лучше другой

SimpleMLP (128→64→1) показал сносный, но не выдающийся результат (R² = 0.886). Сеть слишком мелкая, чтобы выучить нелинейные взаимодействия между 214 признаками — число параметров недостаточно, градиент легко затухает уже ко второму слою.  

DeepMLP с BatchNorm + Dropout (0.3) - антипример. R² всего 0.40, и это объясняется несколькими причинами: агрессивный Dropout (30%) при относительно небольшом датасете (~14 000 обучающих объектов) отключает слишком много нейронов, BatchNorm перед каждым слоем нестабильно взаимодействует с большим Dropout, а CosineLR не успевает довести до хорошего минимума при таком стечении обстоятельств - ранняя остановка срабатывала уже на 54-й эпохе.  

ResidualMLP-S и ResidualMLP-L значительно превзошли все остальные DL-модели. Ключевая причина - остаточные связи: они позволяют градиенту проходить через сеть без затухания. Это особенно важно для табличных данных: многие признаки влияют на цену напрямую (площадь, этаж, класс жилья), и модели выгодно «пробрасывать» их практически без изменений через несколько слоёв. Более широкий dim=512 в ResidualMLP-L даёт больше нейронов для кодирования взаимодействий признаков, что и даёт небольшой, но устойчивый прирост над Small-версией. Активация GELU мягче ReLU и помогает на табличных данных с выбросами — в отличие от жёсткого обнуления.  

TabNet с R² = 0.737 разочаровал. Теоретически attention-based механизм TabNet должен хорошо работать с табличными данными, динамически выбирая важные признаки. На практике это сработало хуже по двум причинам: датасет относительно небольшой, а признаки уже хорошо сформированы и масштабированы - там нет нужды в динамическом отборе признаков, который и является главным козырем архитектуры.

# Почему ML оказался лучше DL в этой задаче

Лучшая DL-модель (ResidualMLP-L: MAE = 5 459, R² = 0.943) почти догнала лучший ML-результат (Random Forest с тюнингом: MAE ≈ 4 899, R² = 0.946), но не превзошла его. Причины в природе задачи:  

- Табличные данные с осмысленными признаками - типичная сильная сторона решающих деревьев. Gradient Boosting и Random Forest умеют эксплуатировать нелинейные пороговые зависимости лучше, чем нейронная сеть, которой нужно выучивать эти пороги из данных через непрерывные веса.
- Размер данных. 17 500 объектов - небольшой датасет для нейросетей. Random Forest с сотнями деревьев извлекает максимум информации из каждого примера через бэггинг. DL-модели начинают уверенно обгонять деревья, как правило, начиная с десятков тысяч объектов и при наличии структурированных пространственно-временных паттернов.
One-hot энкодинг городов и регионов создаёт разреженное пространство признаков с >200 измерениями - решающие деревья работают с такими пространствами нативно, а нейросеть вынуждена вкладывать дополнительную ёмкость в их обработку.
- Отсутствие последовательных и пространственных зависимостей означает отсутствие той структуры, ради которой создавались свёртки и рекуррентные сети. Это чисто табличная задача, где преимущество архитектурных индуктивных смещений DL невозможно реализовать.  


Тем не менее ResidualMLP-L показывает практически сопоставимый результат с Random Forest - это означает, что при наличии большего объёма данных или с применением предобученных табличных трансформеров (FT-Transformer, SAINT) нейросетевой подход может выйти вперёд.